# **ITEM BASED COLLABORATIVE FILTERING**

# Loading Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Importing Datasets

In [ ]:
movies=pd.read_csv('movies.csv')
ratings=pd.read_csv('ratings.csv')

# Movies Dataset

In [ ]:
movies.head(5)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


# Ratings Dataset

In [ ]:
ratings.head(5)

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


# Dropping Genres in Movies

In [ ]:
movies=movies.drop('genres', axis=1)
movies.head(5)

,movieId,title
0,1,Toy Story (1995)
1,2,Jumanji (1995)
2,3,Grumpier Old Men (1995)
3,4,Waiting to Exhale (1995)
4,5,Father of the Bride Part II (1995)


# Dropping Timestamp in Ratings

In [ ]:
ratings=ratings.drop('timestamp', axis=1)
ratings.head(5)

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0


# Merging Movies and Ratings on movieId

In [ ]:
mr=pd.merge(ratings, movies, on='movieId')
mr.head(5)

,userId,movieId,rating,title
0,1,1,4.0,Toy Story (1995)
1,1,3,4.0,Grumpier Old Men (1995)
2,1,6,4.0,Heat (1995)
3,1,47,5.0,Seven (a.k.a. Se7en) (1995)
4,1,50,5.0,"Usual Suspects, The (1995)"


# Dropping movieId from the merged Dataset

In [ ]:
mr=mr.drop('movieId', axis=1)
mr.head(5)

,userId,rating,title
0,1,4.0,Toy Story (1995)
1,1,4.0,Grumpier Old Men (1995)
2,1,4.0,Heat (1995)
3,1,5.0,Seven (a.k.a. Se7en) (1995)
4,1,5.0,"Usual Suspects, The (1995)"


# Creating the User Item Matrix

In [ ]:
mat=mr.pivot_table(index='userId', columns='title', values='rating')
mat.head(5)

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Subtracting User Mean from User Item Matrix

In [ ]:
mean=mat.mean(axis=1)
mat2=mat.sub(mean,axis=0)
mat2.head(5)

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.366379,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Adjusted Cosine Similarity Function

In [ ]:
def acs(target, discount=True):
    if target not in mat2.columns:
       print("Movie not found")
       return
    ii = mat2[target]
    sims = {}
    for col in mat2.columns:
        ij = mat2[col]
        mask = ii.notna() & ij.notna()
        n = mask.sum()
        if n == 0:
            sims[col] = np.nan
            continue
        ir, jr = ii[mask], ij[mask]
        denom = np.linalg.norm(ir) * np.linalg.norm(jr)
        sim = np.dot(ir, jr) / denom if denom else 0
        sims[col] = sim * np.log2(n + 1) if discount else sim
    return pd.Series(sims).sort_values(ascending=False)

# Top Similar Movies Finder Function

In [ ]:
def top(target, user, n=5):
    sims=acs(target)
    un=mat2.loc[user].isna()&(mat2.columns!=target)
    sim=sims[un]
    top5=sim.sort_values(ascending=False).head(n)
    return top5.index.tolist()

# Function to Predict Ratings of Top Similar Movies

In [ ]:
def predict(uid, movies, k=5):
    ur=mat.loc[uid]
    rated=ur[ur.notna()]
    preds={}
    for m in movies:
        if m not in mat2.columns:
            preds[m] = np.nan
            continue
        sim = acs(m)
        sim = sim[rated.index].dropna().sort_values(ascending=False).head(k)
        ratings = rated[sim.index]
        num = (sim * ratings).sum()
        den = sim.abs().sum()
        preds[m] = round(num / den if den else np.nan, 1)
    return pd.Series(preds).sort_values(ascending=False)

# Helper Function for User Input

In [ ]:
def ask():
    user = int(input("Enter User ID: "))
    target = input("Enter Movie Name: ")
    if target not in mat2.columns:
        print("Movie not found")
        return
    if user not in mat2.index:
        print("User not found")
        return
    t5 = top(target, user, 5)
    print("\nRecommended Movies Are:")
    for i in t5:
        print(i)
    print("\nPredicted Ratings of Recommended Movies:")
    print(predict(user,t5).head(5))

# User Input and Recommendations

In [ ]:
ask()

Enter User ID: 1
Enter Movie Name: Toy Story (1995)

Recommended Movies Are:
Toy Story 2 (1999)
Incredibles, The (2004)
Toy Story 3 (2010)
Aladdin (1992)
Finding Nemo (2003)

Predicted Ratings of Recommended Movies:
Toy Story 3 (2010)         4.8
Toy Story 2 (1999)         4.7
Incredibles, The (2004)    4.7
Finding Nemo (2003)        4.6
Aladdin (1992)             4.4
dtype: float64
